# Agentic RAG Security — authorize before ranking

You operate a multi-tenant policy assistant. The agent may reformulate and repeat searches, but it may not select identity, tenant, ACLs, source authority, index generation, or release policy. This notebook is deterministic, credential-free, and makes no model or network call.

## Objectives and safety boundary

We will reproduce rank-then-filter leakage, issue a one-use retrieval grant, prove filtering happens before scoring, preserve exact provenance, reject poisoned and stale evidence, verify claim-level citations, inspect privacy-minimized telemetry, and run the labelled evaluation. Local HMACs and in-memory stores are teaching stand-ins—not production identity, key management, or a vector database.

In [ ]:
from pathlib import Path
import importlib.util, json, sys

def find_course():
    candidates = [Path.cwd(), *Path.cwd().parents]
    relative = Path('curriculum/roadmap/intermediate/08-agentic-rag-security')
    for base in candidates:
        course = base / relative
        if (course / 'lab.py').exists():
            return course
    if (Path.cwd() / 'lab.py').exists():
        return Path.cwd()
    raise FileNotFoundError('course directory not found')

COURSE = find_course()
def load(name, filename):
    spec = importlib.util.spec_from_file_location(name, COURSE / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

lab = load('i08_notebook_lab', 'lab.py')
sdk = load('i08_notebook_sdk', 'sdk_adapter.py')
print('Loaded deterministic lab and Pydantic/OpenTelemetry adapter.')

## Architecture

![Authorization-aware Agentic RAG boundary](architecture.svg)

The purple boxes are proposals or untrusted data. Teal boxes are deterministic decisions. The key ordering is **security trim → rank**, never rank → filter.

## Experiment 1 — reproduce the unsafe baseline

The baseline scores every indexed chunk, selects a global top-k, and filters tenants afterward. A later display filter cannot undo text exposure to the ranker, score service, cache, or telemetry.

In [ ]:
scenario = lab.build_scenario()
identity = scenario.identities.authenticate('attest:north')
unsafe = lab.unsafe_rank_then_filter(
    'Southridge support retention 2555 days', identity, scenario.index, scenario.sources
)
assert unsafe['cross_tenant_scored'] is True
unsafe

The final displayed list can look tenant-safe while the scorer already processed Southridge content. This is why post-ranking filters are not an access-control boundary.

## Experiment 2 — bind a one-use retrieval grant

Trusted runtime state supplies workload, subject, and tenant. The proposal contains only a normalized query, purpose, and bounded top-k.

In [ ]:
scenario = lab.build_scenario()
scenario.start_run('attest:north', 'run:notebook')
issued = scenario.broker.issue(
    'attest:north', request_id='request:notebook:1', run_id='run:notebook',
    proposal=lab.QueryProposal('support case retention period', top_k=4),
)
assert issued.status is lab.DecisionStatus.ALLOW
grant = issued.grant
{
    'tenant': grant.tenant_id, 'subject': grant.subject_id,
    'policy_version': grant.policy_version, 'index_generation': grant.index_generation,
    'top_k': grant.top_k, 'expires_at': grant.expires_at.isoformat(),
    'query_digest_prefix': grant.query_digest[:12],
}

## Experiment 3 — prove policy runs before scoring

The retriever checks tenant, lifecycle, review state, classification, groups, purpose, authority, versions, generation, integrity, and size before it calls the scorer.

In [ ]:
retrieval = scenario.retriever.retrieve('attest:north', grant)
assert retrieval.status is lab.DecisionStatus.ALLOW
scored_sources = {
    chunk.source_id for chunk in scenario.index.chunks
    if chunk.chunk_id in scenario.index.score_calls
}
forbidden = {
    'src:south:retention', 'src:north:submission',
    'src:north:encoded', 'src:north:retention-old',
}
assert scored_sources.isdisjoint(forbidden)
{
    'scored_sources': sorted(scored_sources),
    'evidence_sources': [item.ref.source_id for item in retrieval.evidence],
    'receipt': retrieval.receipt,
}

Neither the obvious poison nor its encoded variant was scored. The invariant comes from source review, purpose, authority, and current registry state—not a phrase detector. Retrieved text that is admitted remains quoted evidence, never instruction authority.

## Experiment 4 — release one exact supported claim

A model could propose the structure below. The application independently checks exact evidence membership, current source state, claim support, authority, lineage, and conflicts.

In [ ]:
proposal = lab.propose_exact_claim(retrieval, 'retention_period')
released = scenario.release_gate.release(
    'attest:north', run_id='run:notebook', retrieval=retrieval, proposal=proposal
)
assert released.status is lab.DecisionStatus.ALLOW
assert released.answer == 'retention_period: 730 days'
released

## Experiment 5 — launder a citation

Changing a cited chunk ID creates a plausible-looking but nonexistent reference. Exact snapshot membership must fail outside the model.

In [ ]:
from dataclasses import replace
claim = proposal.claims[0]
forged_ref = replace(claim.citations[0], chunk_id='chunk:forged')
forged = lab.AnswerProposal((replace(claim, citations=(forged_ref,)),))
blocked = scenario.release_gate.release(
    'attest:north', run_id='run:notebook', retrieval=retrieval, proposal=forged
)
assert (blocked.status, blocked.reason) == (lab.DecisionStatus.DENY, 'citation-not-retrieved')
blocked

## Experiment 6 — freshness and replay

The evidence snapshot records what the model saw, but release still resolves the current source. Separately, a grant can be redeemed only once.

In [ ]:
fresh = lab.build_scenario()
fresh.start_run('attest:north', 'run:freshness')
issued2 = fresh.broker.issue(
    'attest:north', request_id='request:freshness', run_id='run:freshness',
    proposal=lab.QueryProposal('support case retention period', 4),
)
retrieved2 = fresh.retriever.retrieve('attest:north', issued2.grant)
replay = fresh.retriever.retrieve('attest:north', issued2.grant)
source = fresh.sources.sources['src:north:retention']
fresh.sources.sources[source.source_id] = replace(source, version=4)
stale_release = fresh.release_gate.release(
    'attest:north', run_id='run:freshness', retrieval=retrieved2,
    proposal=lab.propose_exact_claim(retrieved2, 'retention_period'),
)
assert replay.reason == 'grant-replayed'
assert stale_release.reason == 'citation-version-or-digest'
{'replay': replay.reason, 'stale_release': stale_release.reason}

## Experiment 7 — strict tool contract and safe telemetry

The real Pydantic schema exposes no tenant or entitlement field. The OpenTelemetry span records a query digest and decision evidence, not the raw query or chunks.

In [ ]:
sdk_report = sdk.demo()
assert sdk_report['input_schema']['additionalProperties'] is False
assert set(sdk_report['input_schema']['properties']) == {'query', 'top_k'}
assert sdk_report['forged_scope_reason'] == 'invalid-tool-input'
assert sdk_report['telemetry_omits_raw_query']
assert sdk_report['telemetry_omits_evidence_text']
{key: sdk_report[key] for key in (
    'safe_status', 'safe_source_ids', 'forged_scope_reason',
    'finished_spans', 'telemetry_has_query_digest', 'model_calls', 'network_calls'
)}

## Evaluation — 36 labelled cases

We keep valid utility, cross-tenant leakage, poison admission, citation behavior, stale release, dependency behavior, and trace completeness separate. A refusal-only system is not counted as useful.

In [ ]:
report, observations = lab.evaluate_controls()
assert (report.cases, report.valid_cases, report.attack_cases, report.failure_cases) == (36, 6, 26, 4)
assert report.valid_completion_rate == 1.0
assert report.cross_tenant_leakage_rate == 0.0
assert report.poisoned_chunk_admission_rate == 0.0
assert report.unsupported_claim_release_rate == 0.0
assert report.stale_evidence_release_rate == 0.0
assert report.dependency_fail_closed_rate == 1.0
report

The unsafe baseline exposure rate is 1.0 by construction; the secure fixture blocks every labelled safety attack while completing every valid case. These are deterministic control tests, not a claim about arbitrary models, vector stores, or corpora. Production evaluation needs larger labelled sets, tenant/source slices, stochastic trials where models are used, and calibrated human review.

## Production upgrade map

| Teaching component | Production replacement |
| --- | --- |
| local attestation registry | workload/user identity plus current group and policy resolution |
| HMAC retrieval grant | protected capability service, managed keys, durable atomic consumption |
| in-memory source registry | governed source catalog and lifecycle/change stream |
| synthetic index | permission-aware vector/search backend with synchronized ACLs and tombstones |
| local score | lexical/dense/hybrid retrieval and reranking after trusted security trimming |
| exact fixture claims | domain rules plus calibrated claim decomposition/entailment and review |
| in-memory spans | protected OTLP pipeline with redaction, access, retention, and alerting |

Measure ACL propagation, index freshness, blocked-valid-query rate, actual forbidden outcomes, claim support, conflict routing, cost per compliant answer, and trace coverage separately.

## Exercises

1. Add a classification change after grant issuance and prove no release occurs.
2. Add two independent sources and enforce a two-lineage threshold.
3. Replace the scorer with BM25 or local embeddings without weakening pre-score filtering.
4. Add a second retrieval step and prove cumulative query/chunk budgets.
5. Add a deletion tombstone and define propagation objectives.
6. Design a `review_required` packet for a high-consequence claim without exposing unrelated evidence.

## Checkpoint and summary

**Question:** An agent supplies a tenant filter and the vector store returns highly relevant chunks with valid embeddings. Is the evidence safe to expose?

**Answer:** No. Authenticated current application state must determine scope, and filtering must precede scoring. Relevance and embedding validity do not establish authorization, provenance, lifecycle, authority, or claim support.

The transferable rule is: **the agent proposes searches and claims; the trusted control plane chooses the searchable universe, preserves exact evidence, and owns release.**